In [ ]:
import numpy as np
import osmnx as ox

def _parse_point(point):
    """Konvertiert Adresse oder Koordinaten in (lat, lon)."""

    # Fall 1: Adresse (String)
    if isinstance(point, str):
        lat, lon = ox.geocode(point)

    # Fall 2: Koordinaten
    elif isinstance(point, (list, tuple, np.ndarray)):
        lat, lon = float(point[0]), float(point[1])

    else:
        raise ValueError(f"{point} muss eine Adresse oder Koordinaten (lat, lon) sein")

    # Sicherheitscheck
    if not (-90 <= lat <= 90 and -180 <= lon <= 180):
        raise ValueError(f"Ungültige Koordinaten: ({lat}, {lon})")

    return lat, lon

In [ ]:
import math

def get_boundingbox_from_points(point1, point2, buffer_km=0.0):
    """
    Erstellt eine Bounding Box aus zwei Punkten und erweitert sie um einen Puffer in Kilometern.
    
    Parameter
    ----------
    point1 : tuple
        Erster Punkt als (lat, lon)
    point2 : tuple
        Zweiter Punkt als (lat, lon)
    buffer_km : float, optional
        Erweiterung der Bounding Box in Kilometern in alle Richtungen (default: 0.0)
    
    Returns
    -------
    tuple
        Bounding Box im Format (north, south, east, west)
    """

    lat1, lon1 = point1
    lat2, lon2 = point2

    # Basis-Bounding-Box
    north = max(lat1, lat2)
    south = min(lat1, lat2)
    east = max(lon1, lon2)
    west = min(lon1, lon2)

    if buffer_km > 0:
        # Umrechnung km -> Grad
        # 1° Latitude ~ 111 km
        delta_lat = buffer_km / 111.0

        # Longitude abhängig von Breitengrad
        mean_lat = (north + south) / 2
        delta_lon = buffer_km / (111.0 * math.cos(math.radians(mean_lat)))

        north += delta_lat
        south -= delta_lat
        east += delta_lon
        west -= delta_lon

    return north, south, east, west

In [ ]:
import os
import json
import uuid
import osmnx as ox


def get_graph_cached(bbox, network_type="drive", size_threshold=0.5, precision=5):
    """
    Lädt einen OSMnx-Graphen aus dem Cache oder erstellt einen neuen basierend auf einer Bounding Box.
    
    Die Funktion prüft, ob bereits ein gespeicherter Graph existiert, dessen Bounding Box
    die angefragte Bounding Box vollständig enthält und dessen Größe innerhalb eines
    definierten Schwellenwerts liegt. Falls ein passender Graph gefunden wird, wird dieser geladen.
    Andernfalls wird ein neuer Graph von OSM heruntergeladen, gespeichert und im Index registriert.
    
    Parameter
    ----------
    bbox : tuple
        Bounding Box im Format (north, south, east, west)
    network_type : str, optional
        Typ des Straßennetzwerks (z.B. "drive", "walk", "bike") (default: "drive")
    size_threshold : float, optional
        Maximal erlaubte relative Größenabweichung zwischen gespeicherter und angefragter Bounding Box.
        Beispiel: 0.5 bedeutet, dass der gespeicherte Graph höchstens 50% größer sein darf (default: 0.5)
    precision : int, optional
        Anzahl Dezimalstellen zur Rundung der Bounding Box Koordinaten (default: 5)
    
    Returns
    -------
    networkx.MultiDiGraph
        Geladener oder neu erstellter OSMnx-Graph
    """

    INDEX_FILE = "data/index.json"
    GRAPH_DIR = "data/graphs"

    os.makedirs(GRAPH_DIR, exist_ok=True)

    # Bounding Box runden
    north, south, east, west = [round(x, precision) for x in bbox]
    requested_area = (north - south) * (east - west)

    # Index laden
    if os.path.exists(INDEX_FILE):
        with open(INDEX_FILE, "r") as f:
            index = json.load(f)
    else:
        index = []

    candidates = []

    for entry in index:
        if entry["network_type"] != network_type:
            continue

        N, S, E, W = entry["north"], entry["south"], entry["east"], entry["west"]

        # Containment
        if not (N >= north and S <= south and E >= east and W <= west):
            continue

        existing_area = (N - S) * (E - W)
        size_ratio = (existing_area - requested_area) / requested_area

        if size_ratio > size_threshold:
            continue

        candidates.append((entry, existing_area))

    # Beste passende Box laden
    if candidates:
        best_entry = min(candidates, key=lambda x: x[1])[0]
        return ox.load_graphml(best_entry["file"])

    # Neuen Graph laden
    G = ox.graph_from_bbox(north, south, east, west, network_type=network_type)

    filename = f"{uuid.uuid4().hex}.graphml"
    filepath = os.path.join(GRAPH_DIR, filename)

    ox.save_graphml(G, filepath)

    # Index erweitern
    index.append({
        "north": north,
        "south": south,
        "east": east,
        "west": west,
        "network_type": network_type,
        "file": filepath
    })

    with open(INDEX_FILE, "w") as f:
        json.dump(index, f, indent=2)

    return G